In [38]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from scipy.stats import skew
from scipy.stats.mstats import winsorize
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold

In [39]:
df = pd.read_csv("Dataset/Harga Bahan Pangan/train/Cabai Rawit Merah.csv")

In [40]:
def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [41]:
df.head()

,Date,Aceh,Bali,Banten,Bengkulu,DI Yogyakarta,DKI Jakarta,Gorontalo,Jambi,Jawa Barat,...,Papua,Riau,Sulawesi Barat,Sulawesi Selatan,Sulawesi Tengah,Sulawesi Tenggara,Sulawesi Utara,Sumatera Barat,Sumatera Selatan,Sumatera Utara
0,2022-01-01,NaN,76680.0,80020.0,43040.0,63490.0,98730.0,64110.0,51740.0,91430.0,...,105460.0,60000.0,72680.0,66000.0,84750.0,83440.0,98580.0,NaN,68790.0,55000.0
1,2022-01-02,NaN,74370.0,81700.0,45830.0,64110.0,98730.0,63440.0,59810.0,93530.0,...,100990.0,60000.0,67780.0,64250.0,82800.0,77920.0,90400.0,NaN,68220.0,68410.0
2,2022-01-03,NaN,72660.0,84630.0,53330.0,70680.0,97420.0,67340.0,66470.0,86060.0,...,96970.0,65000.0,68490.0,62610.0,61150.0,72530.0,89100.0,NaN,71500.0,74700.0
3,2022-01-04,NaN,75000.0,88680.0,53840.0,69670.0,95730.0,63990.0,65410.0,88960.0,...,93580.0,NaN,64210.0,60620.0,78500.0,72470.0,79590.0,NaN,66170.0,70360.0
4,2022-01-05,50000.0,70020.0,79400.0,53840.0,63250.0,88960.0,62670.0,64750.0,87970.0,...,98180.0,65000.0,65310.0,60440.0,75770.0,73930.0,76580.0,NaN,66420.0,70360.0


In [42]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                          0.000000
Aceh                         77.888446
Bali                          3.585657
Banten                        3.685259
Bengkulu                      3.685259
DI Yogyakarta                 3.585657
DKI Jakarta                   3.685259
Gorontalo                     3.486056
Jambi                         3.784861
Jawa Barat                    3.685259
Jawa Tengah                   3.386454
Jawa Timur                    3.486056
Kalimantan Barat              3.585657
Kalimantan Selatan            3.685259
Kalimantan Tengah             3.585657
Kalimantan Timur              3.884462
Kalimantan Utara              3.884462
Kepulauan Bangka Belitung     3.784861
Kepulauan Riau                3.884462
Lampung                       3.685259
Maluku Utara                  3.585657
Maluku                        3.685259
Nusa Tenggara Barat           3.685259
Nusa Tenggara Timur           3.386454
Papua Barat                   3.884462
Papua                    

In [43]:
numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].fillna(df[numeric_features].median())

In [44]:
zero_var_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with zero variance:", zero_var_cols)

Columns with zero variance: []


In [45]:
missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

Date                         0.0
Aceh                         0.0
Bali                         0.0
Banten                       0.0
Bengkulu                     0.0
DI Yogyakarta                0.0
DKI Jakarta                  0.0
Gorontalo                    0.0
Jambi                        0.0
Jawa Barat                   0.0
Jawa Tengah                  0.0
Jawa Timur                   0.0
Kalimantan Barat             0.0
Kalimantan Selatan           0.0
Kalimantan Tengah            0.0
Kalimantan Timur             0.0
Kalimantan Utara             0.0
Kepulauan Bangka Belitung    0.0
Kepulauan Riau               0.0
Lampung                      0.0
Maluku Utara                 0.0
Maluku                       0.0
Nusa Tenggara Barat          0.0
Nusa Tenggara Timur          0.0
Papua Barat                  0.0
Papua                        0.0
Riau                         0.0
Sulawesi Barat               0.0
Sulawesi Selatan             0.0
Sulawesi Tengah              0.0
Sulawesi T

In [46]:
numerical_features = df.select_dtypes(include=['number'])
categorical_features = df.select_dtypes(exclude=['number'])

# Apply VarianceThreshold to remove low-variance numerical features
selector = VarianceThreshold(threshold=0.01)  # Adjust threshold as needed
reduced_numerical_df = selector.fit_transform(numerical_features)

# Convert back to DataFrame with selected features
reduced_numerical_df = pd.DataFrame(reduced_numerical_df, 
                                       columns=numerical_features.columns[selector.get_support()])

# Combine numerical and categorical features back together
reduced_df = pd.concat([reduced_numerical_df, categorical_features.reset_index(drop=True)], axis=1)

In [47]:
df.shape

(1004, 35)

In [48]:
reduced_df.shape

(1004, 35)

In [49]:
skewness = df.select_dtypes(include=['number']).apply(lambda x: stats.skew(x.dropna())).sort_values(ascending=False)
print(skewness)

Sulawesi Tengah              1.940210
Sulawesi Tenggara            1.743209
Sulawesi Utara               1.704129
Sulawesi Selatan             1.454056
Kalimantan Utara             1.427560
Maluku Utara                 1.386954
Gorontalo                    1.356154
Sulawesi Barat               1.288208
Kepulauan Riau               1.212636
Kepulauan Bangka Belitung    1.143789
Lampung                      1.074860
Kalimantan Timur             1.008127
Jawa Tengah                  0.954608
Jambi                        0.930344
DKI Jakarta                  0.904277
Papua Barat                  0.891123
Jawa Timur                   0.887507
Jawa Barat                   0.882369
Sumatera Selatan             0.855326
DI Yogyakarta                0.833074
Kalimantan Selatan           0.809966
Kalimantan Tengah            0.772349
Banten                       0.769196
Bengkulu                     0.734582
Bali                         0.734206
Riau                         0.696587
Sumatera Bar

In [50]:
# Select numerical columns
num_cols = df.select_dtypes(include=['number'])

# Function to calculate outlier percentage using IQR
def outlier_percentage(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((column < lower_bound) | (column > upper_bound)).sum()
    return (outliers / len(column)) * 100  # Percentage

# Apply function to all numerical columns
outlier_percentages_df = num_cols.apply(outlier_percentage)

# Display the results
print(outlier_percentages_df.sort_values(ascending=False))

Aceh                         21.713147
Sulawesi Tenggara             5.179283
Sulawesi Selatan              4.780876
Sumatera Utara                4.780876
Kalimantan Utara              3.984064
Kepulauan Bangka Belitung     3.784861
Sulawesi Tengah               3.685259
Sulawesi Barat                3.585657
Gorontalo                     3.187251
Bengkulu                      3.187251
Maluku Utara                  3.087649
Sulawesi Utara                3.087649
Papua Barat                   2.788845
Jambi                         2.589641
Sumatera Barat                2.589641
Kalimantan Timur              2.091633
Riau                          1.992032
Kepulauan Riau                1.892430
Sumatera Selatan              1.792829
Kalimantan Selatan            1.693227
Lampung                       1.394422
Papua                         1.394422
DKI Jakarta                   1.195219
Kalimantan Tengah             1.095618
Kalimantan Barat              0.896414
DI Yogyakarta            

In [51]:
test = pd.read_csv("Dataset/Harga Bahan Pangan/test/Cabai Rawit Merah.csv")

In [52]:
df.to_csv("Cabai Rawit Merah Clean.csv", index=False)

In [53]:
def df_to_X_y(df, window_size=5):
  df_as_np = df.to_numpy()
  X = []
  y = []
  for i in range(len(df_as_np)-window_size):
    row = [[a] for a in df_as_np[i:i+window_size]]
    X.append(row)
    label = df_as_np[i+window_size]
    y.append(label)
  return np.array(X), np.array(y)

In [54]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping


In [55]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, InputLayer, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split

# Load dataset
df = df.select_dtypes(include=[np.number])  # Keep only numeric columns
df = df.apply(pd.to_numeric, errors='coerce').dropna()  # Convert and drop NaNs

# If multiple numeric columns exist, use the first one
if df.shape[1] > 1:
    print(f"Warning: DataFrame has multiple numeric columns ({df.shape[1]}). Using the first column.")
    df = df.iloc[:, 0]

# Convert data into sequences
def df_to_X_y(df, window_size=10):  # Keep window size = 10 for CNN effectiveness
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size].values)  # Directly use raw values
        y.append(df.iloc[i + window_size])  # Keep original values
    return np.array(X), np.array(y)

X, y = df_to_X_y(df, window_size=10)

# Reshape for CNN-LSTM (Conv1D requires 3D input)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# Callbacks
checkpoint_path = "model_checkpoint.keras"
cp4 = ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_loss', mode='min')
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.8, patience=5, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# Define CNN-LSTM model
def create_cnn_lstm_model(input_shape):
    model = Sequential([
        InputLayer(input_shape=input_shape),

        # CNN Layers
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),

        # LSTM Layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),

        # Dense Layers
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # Output remains in original scale
    ])
    
    model.compile(loss='mape', optimizer=Adam(learning_rate=0.003), metrics=['mape'])
    return model

# Train the model
input_shape = (X_train.shape[1], X_train.shape[2])
model = create_cnn_lstm_model(input_shape)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
          epochs=300, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Training completed. Final epoch:", len(history.history['loss']))

# Save the model
model.save("cnn_lstm_model.keras")

# Forecasting
df_submission = pd.read_csv("Dataset/Harga Bahan Pangan/sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, input_seq.shape[0], 1))[0, 0]
    pred += np.random.normal(0, 0.01)  # Add small noise to prevent stagnation
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

# Check if predictions match expected count
if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

# Save submission file
data = [{'id': df_submission.iloc[i]['id'], 'price': future_predictions[i]} for i in range(min(len(df_submission), len(future_predictions)))]
submission_df = pd.DataFrame(data)
submission_df.to_csv("Cabai_Rawit_Merah_cnn_lstm_submission.csv", index=False)
print("Submission file saved as Cabai_Rawit_Merah_cnn_lstm_submission.csv")

d:\Anaconda\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning:

Argument `input_shape` is deprecated. Use `shape` instead.



Epoch 1/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 65ms/step - loss: 99.9970 - mape: 99.9970 - val_loss: 99.9840 - val_mape: 99.9840 - learning_rate: 0.0030
Epoch 2/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 99.9786 - mape: 99.9786 - val_loss: 99.9599 - val_mape: 99.9599 - learning_rate: 0.0030
Epoch 3/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 99.9524 - mape: 99.9524 - val_loss: 99.9264 - val_mape: 99.9264 - learning_rate: 0.0030
Epoch 4/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 99.9162 - mape: 99.9162 - val_loss: 99.8824 - val_mape: 99.8824 - learning_rate: 0.0030
Epoch 5/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 99.8685 - mape: 99.8685 - val_loss: 99.8273 - val_mape: 99.8273 - learning_rate: 0.0030
Epoch 6/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 99.8103 - mape: 99.8103 - val_loss: 99.7609 - val_mape: 99.7609 - learning_rate: 0.0030
Epoch 7/300
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 99.7406 - mape: 99.7406 - val_loss: 99.6826 - val_m